In [1]:
import sqlite3
from pathlib import Path

DB_PATH = Path("..") / "database" / "ans.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print("Conectado em:", DB_PATH.resolve())

Conectado em: C:\Users\Uélinton\jupyter_projects\ans_market_analytics\database\ans.db


In [2]:
query = """
WITH operadora_mes AS (
    SELECT
        b.competencia,
        b.registro_ans,
        MAX(b.beneficiarios) AS beneficiarios
    FROM si_beneficiarios_operadora b
    GROUP BY
        b.competencia,
        b.registro_ans
),
market_share AS (
    SELECT
        competencia,
        registro_ans,
        beneficiarios,
        (beneficiarios * 100.0 / SUM(beneficiarios) OVER (PARTITION BY competencia)) AS market_share_pct
    FROM operadora_mes
)
SELECT
    competencia,
    ROUND(SUM(market_share_pct * market_share_pct), 2) AS hhi
FROM market_share
GROUP BY competencia
ORDER BY competencia;
"""

cursor.execute(query)
rows = cursor.fetchall()

for row in rows:
    print(row)

('2025-12-01', 207.2)


In [3]:
query = """
WITH operadora_mes AS (
    SELECT
        b.competencia,
        b.registro_ans,
        MAX(b.beneficiarios) AS beneficiarios
    FROM si_beneficiarios_operadora b
    GROUP BY
        b.competencia,
        b.registro_ans
),
market_share AS (
    SELECT
        competencia,
        registro_ans,
        beneficiarios,
        (beneficiarios * 100.0 / SUM(beneficiarios) OVER (PARTITION BY competencia)) AS market_share_pct
    FROM operadora_mes
),
hhi_calc AS (
    SELECT
        competencia,
        ROUND(SUM(market_share_pct * market_share_pct), 2) AS hhi
    FROM market_share
    GROUP BY competencia
)
SELECT
    competencia,
    hhi,
    CASE
        WHEN hhi < 1500 THEN 'Mercado pouco concentrado'
        WHEN hhi BETWEEN 1500 AND 2500 THEN 'Mercado moderadamente concentrado'
        ELSE 'Mercado altamente concentrado'
    END AS classificacao
FROM hhi_calc
ORDER BY competencia;
"""

cursor.execute(query)
rows = cursor.fetchall()

for row in rows:
    print(row)

('2025-12-01', 207.2, 'Mercado pouco concentrado')


In [4]:
query = """
WITH operadora_mes AS (
    SELECT
        b.competencia,
        b.registro_ans,
        o.razao_social,
        MAX(b.beneficiarios) AS beneficiarios
    FROM si_beneficiarios_operadora b
    LEFT JOIN si_operadora o
        ON o.registro_ans = b.registro_ans
    GROUP BY
        b.competencia,
        b.registro_ans,
        o.razao_social
),
market_share AS (
    SELECT
        competencia,
        registro_ans,
        razao_social,
        beneficiarios,
        ROUND(
            beneficiarios * 100.0 /
            SUM(beneficiarios) OVER (PARTITION BY competencia),
            4
        ) AS market_share_pct,
        RANK() OVER (
            PARTITION BY competencia
            ORDER BY beneficiarios DESC
        ) AS ranking
    FROM operadora_mes
)
SELECT
    competencia,
    ranking,
    registro_ans,
    razao_social,
    beneficiarios,
    market_share_pct,
    ROUND(
        SUM(market_share_pct) OVER (
            PARTITION BY competencia
            ORDER BY ranking
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ),
        4
    ) AS market_share_acumulado_pct
FROM market_share
WHERE ranking <= 10
ORDER BY competencia, ranking;
"""

cursor.execute(query)
rows = cursor.fetchall()

for row in rows:
    print(row)

('2025-12-01', 1, '301949', 'ODONTOPREV S/A', 6510431, 9.5755, 9.5755)
('2025-12-01', 2, '359017', 'NOTRE DAME INTERMÉDICA SAÚDE S.A.', 2982188, 4.3862, 13.9617)
('2025-12-01', 3, '368253', 'HAPVIDA ASSISTENCIA MEDICA S.A.', 2979226, 4.3818, 18.3435)
('2025-12-01', 4, '005711', 'BRADESCO SAÚDE S.A.', 2828475, 4.1601, 22.5036)
('2025-12-01', 5, '326305', 'AMIL ASSISTÊNCIA MÉDICA INTERNACIONAL S.A.', 2574733, 3.7869, 26.2905)
('2025-12-01', 6, '006246', 'SUL AMERICA COMPANHIA DE SEGURO SAÚDE', 2164765, 3.1839, 29.4744)
('2025-12-01', 7, '339679', 'UNIMED NACIONAL - COOPERATIVA CENTRAL', 1367180, 2.0108, 31.4852)
('2025-12-01', 8, '374440', 'PREVIDENT ASSISTÊNCIA ODONTOLÓGICA S.A', 1249838, 1.8382, 33.3234)
('2025-12-01', 9, '000582', 'PORTO SEGURO - SEGURO SAÚDE S/A', 1151918, 1.6942, 35.0176)
('2025-12-01', 10, '343889', 'UNIMED BELO HORIZONTE COOPERATIVA DE TRABALHO MÉDICO', 1075278, 1.5815, 36.5991)


## Interpretação do HHI

O índice HHI mede o nível de concentração do mercado a partir da soma dos quadrados das participações de mercado das operadoras.

Faixas de interpretação:
- abaixo de 1500: mercado pouco concentrado
- entre 1500 e 2500: mercado moderadamente concentrado
- acima de 2500: mercado altamente concentrado

No contexto da saúde suplementar, esse indicador ajuda a entender se o setor é pulverizado ou dominado por poucos grandes grupos.

In [5]:
conn.close()